# 🔬 Notebook 3: Collaborative Whiteboard — Deep Dive


## 🛠️ Setup

```bash
cd 06-system-designs/collaborative-whiteboard
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🗺️ What this notebook covers

We take several ideas and implement a tiny, runnable version of each:

1. **🏚️ Naive concurrency** — why a straightforward "last message wins" server loses edits.
2. **🏛️ Last-Writer-Wins CRDT** — how Lamport + actor tie-break gives deterministic convergence.
3. **🔀 Offline merge** — two users edit while partitioned and the state still converges.
4. **📣 Pub/Sub fan-out** — how WebSocket servers share ops across nodes.
5. **📸 Snapshots + op log** — how late joiners catch up fast.
6. **👻 Presence with TTL** — ephemeral cursors, never persisted.

Everything below runs in-process — no external services.


## 1️⃣ 🏚️ The naive server — and how it loses edits

Imagine the simplest possible server: when a client sends an op, the server just overwrites the shape in an in-memory dict, then broadcasts. No clocks, no merging.

This works *if ops arrive in the same order on every node*. The moment there's any reordering (and there always is — Wi-Fi blips, sharded gateways, mobile networks), edits are **lost**.


In [ ]:
# Naive: just apply ops in the order they arrive, no clocks.
class NaiveBoard:
    def __init__(self):
        self.shapes = {}
    def apply(self, op):
        if op["kind"] == "delete":
            self.shapes.pop(op["shape_id"], None)
        else:
            self.shapes[op["shape"]["id"]] = op["shape"]

# Two users edit the same shape "concurrently".
# Server A sees alice's ops first; Server B sees bob's first.
ops_A = [
    {"kind":"add",    "shape":{"id":"s1","x":0,"color":"red"}},
    {"kind":"update", "shape":{"id":"s1","x":10,"color":"red"}},   # alice drags
    {"kind":"update", "shape":{"id":"s1","x":10,"color":"blue"}},  # bob recolors
]
ops_B = [
    {"kind":"add",    "shape":{"id":"s1","x":0,"color":"red"}},
    {"kind":"update", "shape":{"id":"s1","x":10,"color":"blue"}},  # bob first here
    {"kind":"update", "shape":{"id":"s1","x":10,"color":"red"}},   # alice arrives later
]

a, b = NaiveBoard(), NaiveBoard()
for o in ops_A: a.apply(o)
for o in ops_B: b.apply(o)
print("Server A sees:", a.shapes["s1"])
print("Server B sees:", b.shapes["s1"])
print("Diverged! Clients on A see blue, clients on B see red.")


👉 The naive design **cannot converge** under reordering. We need two things:

1. A **logical timestamp** on every op (Lamport).
2. A **merge rule** that's associative and commutative — same ops, any order, same state.

That's exactly what a CRDT gives us.


## 2️⃣ 🏛️ Last-Writer-Wins (LWW) Map CRDT

Model the board as `shape_id → (timestamp, shape_or_tombstone)`. When two ops touch the same shape, the one with the **higher Lamport** wins. Tie → break on `actor` id (any consistent rule works).

Key property: for any set of ops, **any order** of `apply` yields the **same** final state. That's *convergence*.


In [ ]:
from dataclasses import dataclass

@dataclass(order=True, frozen=True)
class Ts:
    lamport: int
    actor: str   # tie-breaker; any stable total order works

class LWWBoard:
    """Board as map shape_id -> (Ts, shape | None). None = tombstone."""
    def __init__(self):
        self.state: dict[str, tuple[Ts, dict | None]] = {}

    def apply(self, op: dict):
        ts = Ts(op["lamport"], op["actor"])
        sid = op["shape_id"] if op["kind"] == "delete" else op["shape"]["id"]
        cur = self.state.get(sid)
        if cur is None or ts > cur[0]:
            if op["kind"] == "delete":
                self.state[sid] = (ts, None)
            else:
                self.state[sid] = (ts, op["shape"])

    def shapes(self) -> list[dict]:
        return [s for _, s in self.state.values() if s is not None]


ops = [
    {"kind":"add",    "shape":{"id":"s1","x":0,"color":"red"},    "lamport":1, "actor":"alice"},
    {"kind":"update", "shape":{"id":"s1","x":10,"color":"red"},   "lamport":3, "actor":"alice"},
    {"kind":"update", "shape":{"id":"s1","x":10,"color":"blue"},  "lamport":3, "actor":"bob"},    # tie on lamport, bob > alice
    {"kind":"update", "shape":{"id":"s1","x":20,"color":"blue"},  "lamport":4, "actor":"bob"},
]

import random
random.seed(0)
# Try several permutations — they must all converge to the same state.
last = None
for perm in [ops, list(reversed(ops)), random.sample(ops, len(ops)), random.sample(ops, len(ops))]:
    b = LWWBoard()
    for o in perm: b.apply(o)
    final = b.shapes()
    print("order -> final:", final)
    if last is not None:
        assert final == last, "diverged!"
    last = final
print("\nEvery order converged to the same state.")


### Why does this work?

LWW on a single key is a **last-write-wins register**, a classic CRDT. Promoting it to a *map* of such registers (one per `shape_id`) keeps the CRDT property. Deletes are **tombstones** (the `None` value) so a late-arriving `add` doesn't resurrect a deleted shape.

> ⚖️ **Why LWW-CRDT and not OT?** We worked that decision out with runnable code in
> [notebook 1](./01_requirements_and_architecture.ipynb) — including a working OT transform,
> the N² growth of its rule set, and the five things this choice costs us. The short version:
> our unit of edit is a whole shape (not a character in a sequence), offline is a hard
> requirement, and one merge rule beats 49 transform rules. The bill arrives in §3.1 below.


## 3️⃣ 🔀 Offline merge — both users edit while partitioned

This is the *real* test. Alice's laptop loses Wi-Fi on the train. She keeps editing. Bob keeps editing online. When Alice reconnects, both op streams are merged. The final board should be what you'd get if their edits had been interleaved all along.


In [ ]:
# Shared starting point
common = [
    {"kind":"add", "shape":{"id":"s1","x":0, "color":"red"},   "lamport":1, "actor":"alice"},
    {"kind":"add", "shape":{"id":"s2","x":0, "color":"green"}, "lamport":2, "actor":"bob"},
]

# While partitioned, each side ticks its own Lamport starting from 2 (last observed).
alice_offline = [
    {"kind":"update", "shape":{"id":"s1","x":50,"color":"red"},    "lamport":3, "actor":"alice"},
    {"kind":"delete", "shape_id":"s2",                             "lamport":4, "actor":"alice"},
]
bob_online = [
    {"kind":"update", "shape":{"id":"s1","x":0, "color":"purple"}, "lamport":3, "actor":"bob"},    # conflicts with alice's move
    {"kind":"add",    "shape":{"id":"s3","x":99,"color":"yellow"}, "lamport":4, "actor":"bob"},
]

def run(all_ops):
    b = LWWBoard()
    for o in all_ops: b.apply(o)
    return sorted(b.shapes(), key=lambda s: s["id"])

alice_view = run(common + alice_offline + bob_online)
bob_view   = run(common + bob_online + alice_offline)   # different interleaving

print("alice final:", alice_view)
print("bob final:  ", bob_view)
assert alice_view == bob_view
print("\n✔ Converged: both interleavings produce byte-identical state.")

# ── Now look closely at s1. Convergence is not the same as correctness. ──
s1 = next(s for s in alice_view if s["id"] == "s1")
print(f"\ns1 = {s1}")
print("  alice moved it to x=50 while offline  ->", "KEPT" if s1["x"] == 50 else "LOST")
print("  bob recoloured it to purple           ->", "KEPT" if s1["color"] == "purple" else "LOST")
print("\n⚠️  Whole-shape LWW threw one of them away. Nobody was told. Alice will reconnect,")
print("    watch her rectangle snap back to where it was, and file a bug.")


### 3.1 ⚠️ Convergence is not correctness — the lost update

Every replica agrees, which is what a CRDT promises. But Alice's **move** and Bob's
**recolour** touched *different properties of the same shape*, and whole-shape LWW keeps
exactly one whole shape — so one of two perfectly compatible edits silently vanished.

This is the single most common complaint about LWW in real products, and it is not a bug in
the implementation; it is the granularity of the register being wrong. The fix is to shrink
the register: make the CRDT a **map of per-field LWW registers** instead of one register
holding the whole shape.

In [ ]:
class FieldLWWBoard:
    """Board as shape_id -> {field: (Ts, value)}. Each FIELD is its own LWW register."""
    def __init__(self):
        self.state: dict[str, dict[str, tuple]] = {}
        self.deleted: dict[str, Ts] = {}      # shape_id -> tombstone Ts

    def apply(self, op: dict):
        ts = Ts(op["lamport"], op["actor"])
        if op["kind"] == "delete":
            sid = op["shape_id"]
            cur = self.deleted.get(sid)
            if cur is None or ts > cur:
                self.deleted[sid] = ts
            return
        sid = op["shape"]["id"]
        fields = self.state.setdefault(sid, {})
        # Only the fields this op actually carries compete for the write.
        for k, v in op["shape"].items():
            if k == "id":
                continue
            cur = fields.get(k)
            if cur is None or ts > cur[0]:
                fields[k] = (ts, v)

    def shapes(self) -> list[dict]:
        out = []
        for sid, fields in self.state.items():
            tomb = self.deleted.get(sid)
            # A delete wins only against writes that are older than it.
            if tomb and all(tomb > ts for ts, _ in fields.values()):
                continue
            out.append({"id": sid, **{k: v for k, (_, v) in fields.items()}})
        return sorted(out, key=lambda s: s["id"])


# Same story, but each op now carries ONLY the fields it changed — a real client
# sends a patch ("I moved it"), not a whole shape ("here is my idea of the shape").
common_f = [
    {"kind": "add", "shape": {"id": "s1", "x": 0, "color": "red"},   "lamport": 1, "actor": "alice"},
    {"kind": "add", "shape": {"id": "s2", "x": 0, "color": "green"}, "lamport": 2, "actor": "bob"},
]
alice_offline_f = [
    {"kind": "update", "shape": {"id": "s1", "x": 50},                "lamport": 3, "actor": "alice"},
    {"kind": "delete", "shape_id": "s2",                              "lamport": 4, "actor": "alice"},
]
bob_online_f = [
    {"kind": "update", "shape": {"id": "s1", "color": "purple"},      "lamport": 3, "actor": "bob"},
    {"kind": "add",    "shape": {"id": "s3", "x": 99, "color": "yellow"}, "lamport": 4, "actor": "bob"},
]

def run_f(all_ops):
    b = FieldLWWBoard()
    for o in all_ops: b.apply(o)
    return b.shapes()

a_view = run_f(common_f + alice_offline_f + bob_online_f)
b_view = run_f(common_f + bob_online_f + alice_offline_f)
assert a_view == b_view, "per-field LWW must still converge"

s1 = next(s for s in a_view if s["id"] == "s1")
print("s1 =", s1)
print("  alice's move      ->", "KEPT" if s1["x"] == 50 else "LOST")
print("  bob's recolour    ->", "KEPT" if s1["color"] == "purple" else "LOST")
print("  s2 still deleted  ->", all(s["id"] != "s2" for s in a_view))
print("  s3 present        ->", any(s["id"] == "s3" for s in a_view))
print("\n✔ Still converges, and now BOTH edits survive.")

# What it cost us. Price it on a realistic shape, not on our 2-field toy.
TS_BYTES = 16                               # lamport (8) + actor id (8)
real_fields = ["x", "y", "w", "h", "color", "stroke", "rotation", "z"]
print(f"\ncost on a realistic {len(real_fields)}-field shape:")
print(f"  whole-shape LWW: 1 timestamp  = {TS_BYTES:>3} B/shape")
print(f"  per-field  LWW: {len(real_fields)} timestamps = {len(real_fields)*TS_BYTES:>3} B/shape")
print(f"  on a 100k-shape board that is "
      f"{(len(real_fields)-1)*TS_BYTES*100_000/1e6:.1f} MB of extra metadata, in RAM and in every snapshot")

⚖️ **Per-field LWW is not free, and it is not always right:**

- **Metadata multiplies.** One timestamp per shape becomes one per *field*. On a board with
  100k shapes × 8 fields that is real memory, and it is memory you also have to ship in the
  snapshot.
- **It can produce states no user ever authored.** Alice drags a shape to the left while Bob
  resizes it; per-field merge happily keeps Alice's `x` and Bob's `w`, producing a rectangle
  neither of them made. Sometimes that is exactly right (independent properties); sometimes
  it is nonsense (`x`/`y` of a drag are one intention and should move together). The
  practical answer is to group co-dependent fields into one register — `(x, y)` as a unit,
  `(w, h)` as a unit — which is a judgement call about *your* data, not a general rule.
- **Delete vs update stays awkward.** Our `shapes()` keeps a shape alive if *any* field write
  is newer than the tombstone. That is "delete loses to a concurrent edit", which is a
  defensible default (data loss is worse than a resurrected shape) but it is a *policy*, and
  the opposite policy is equally defensible. Pick one deliberately and write it down.

## 4️⃣ 📣 Pub/Sub fan-out across WebSocket nodes

In production, WebSocket gateways are stateless and many boards span multiple nodes. Pub/sub (Redis, NATS, Kafka…) is how nodes share ops.

Below is a toy in-process bus so you can see the shape of the code.


In [ ]:
from collections import defaultdict

class Bus:
    """Tiny synchronous pub/sub. In prod: Redis PUBSUB, NATS, or Kafka."""
    def __init__(self):
        self._subs = defaultdict(list)
    def subscribe(self, topic, fn):
        self._subs[topic].append(fn)
    def publish(self, topic, msg):
        for fn in self._subs[topic]:
            fn(msg)


class WSGateway:
    """Simulates one WS gateway node holding some client connections."""
    def __init__(self, name, bus):
        self.name = name
        self.clients = defaultdict(list)  # board_id -> list[client_name]
        bus.subscribe("board:b1", self._on_msg)

    def attach(self, board_id, client_name):
        self.clients[board_id].append(client_name)

    def _on_msg(self, op):
        # Fan out to our local sockets only — no wasted work.
        for c in self.clients.get(op["board_id"], []):
            payload = op.get("shape", op.get("shape_id"))
            print(f"  [{self.name}] -> {c}: {op['kind']} {payload}")


bus = Bus()
node_A = WSGateway("node-A", bus); node_A.attach("b1", "alice")
node_B = WSGateway("node-B", bus); node_B.attach("b1", "bob"); node_B.attach("b1", "carol")

# Alice sends an op to node-A. node-A validates, assigns lamport, and publishes.
alice_op = {"board_id":"b1","actor":"alice","lamport":5,"kind":"add","shape":{"id":"s9","x":0}}
print("Alice edits. Broadcasting via pub/sub:")
bus.publish("board:b1", alice_op)


### Gotchas with pub/sub
- **Echo suppression** — Alice shouldn't re-render her own op. Filter by `actor` on the client.
- **At-most-once vs at-least-once** — Redis Pub/Sub can drop messages during failover. Safe pattern: **durable op log + pub/sub as cache**. On reconnect, clients ask the op log for anything past their `since` cursor.
- **Room affinity** — shard by `board_id` (consistent hash) to keep a board's traffic on fewer nodes and reduce fan-out.


## 5️⃣ 📸 Snapshots + op log — fast catch-up for late joiners

Replaying a million ops to join a meeting would be silly (notebook 1 sized the log at
terabytes per day). Every N ops, compress the state to a **snapshot**. Late joiners download
the snapshot and replay only the tail.

Watch the two cursors below. It is very tempting to use one number for both "how many ops
have we snapshotted" and "what lamport is the snapshot valid to" — they are equal in a
single-actor toy and diverge immediately with two actors. That bug ships surprisingly often
and shows up as late joiners missing the last few seconds of edits.

In [ ]:
class OpLog:
    def __init__(self):
        self.ops = []                       # append-only
    def append(self, op): self.ops.append(op)
    def since(self, lamport): return [o for o in self.ops if o["lamport"] > lamport]

class Snapshotter:
    """Compress the log to a full state every `every` ops.

    Note we track TWO different cursors and they are NOT interchangeable:
      * snapshot_index   -- how many ops we had consumed (a count, drives the trigger)
      * snapshot_lamport -- the logical clock the snapshot is valid up to (drives catch-up)
    Conflating them works only while lamport == index, which is true in this toy and
    false the moment two actors are involved.
    """
    def __init__(self, every=3):
        self.every = every
        self.snapshot = None
        self.snapshot_index = 0
        self.snapshot_lamport = 0

    def maybe_snapshot(self, log: OpLog):
        if len(log.ops) - self.snapshot_index < self.every:
            return
        b = LWWBoard()
        for o in log.ops:
            b.apply(o)
        self.snapshot = b.shapes()
        self.snapshot_index = len(log.ops)
        self.snapshot_lamport = max(o["lamport"] for o in log.ops)
        print(f"snapshot @ lamport={self.snapshot_lamport} (after {self.snapshot_index} ops), "
              f"{len(self.snapshot)} shapes")


log = OpLog(); snap = Snapshotter(every=3)
# Two actors, so lamport values collide and no longer equal the op index.
for i in range(1, 8):
    for actor in ("alice", "bob"):
        log.append({"board_id": "b1", "actor": actor, "lamport": i, "kind": "add",
                    "shape": {"id": f"s{i}-{actor}", "x": i * 10, "color": "black"}})
    snap.maybe_snapshot(log)

tail = log.since(snap.snapshot_lamport)
print(f"\nlog has {len(log.ops)} ops; index cursor={snap.snapshot_index}, "
      f"lamport cursor={snap.snapshot_lamport}  <- different numbers, different jobs")
print(f"Late joiner: download snapshot ({len(snap.snapshot)} shapes) + replay {len(tail)} tail ops")

# The catch-up must reproduce the same board as a full replay from op 0.
full = LWWBoard()
for o in log.ops: full.apply(o)
caught_up = LWWBoard()
for o in log.ops[:snap.snapshot_index]: caught_up.apply(o)   # stands in for loading the snapshot
for o in tail: caught_up.apply(o)
assert sorted(full.shapes(), key=lambda s: s["id"]) == sorted(caught_up.shapes(), key=lambda s: s["id"])
print("✔ snapshot + tail == full replay")

## 6️⃣ 👻 Presence with TTL — ephemeral cursors

Cursors move 60 times per second. Persisting them would drown any database. Instead:

- Presence flows through pub/sub only.
- Each user's presence has a **TTL** (e.g., 10s). If no heartbeat, drop it.
- On disconnect, publish a `presence_gone` so peers can hide the cursor immediately.


In [ ]:
import time

class PresenceRoom:
    def __init__(self, ttl_sec=10.0):
        self.ttl = ttl_sec
        self.cursors = {}  # actor -> (x, y, last_seen)

    def update(self, actor, x, y, now=None):
        self.cursors[actor] = (x, y, now or time.time())

    def active(self, now=None):
        now = now or time.time()
        return {a:(x,y) for a,(x,y,t) in self.cursors.items() if now - t <= self.ttl}


room = PresenceRoom(ttl_sec=2.0)
t0 = 1000.0
room.update("alice", 10, 20, now=t0)
room.update("bob",   30, 40, now=t0 + 0.5)
print("t0+1s  active:", room.active(now=t0 + 1.0))   # both
print("t0+3s  active:", room.active(now=t0 + 3.0))   # both expired
room.update("alice", 11, 21, now=t0 + 3.0)           # heartbeat
print("t0+3.1 active:", room.active(now=t0 + 3.1))   # only alice


## 🧠 Closing thoughts

- **CRDTs** (here: LWW map) remove the need for a central serializer → offline-friendly, partition-tolerant.
- **Lamport + actor** is the simplest stable total order that gives deterministic convergence.
- **Snapshots** keep the op log from becoming a liability.
- **Pub/Sub** is the *broadcast mechanism*, not the *source of truth*. The durable op log is the source of truth.
- **Presence is ephemeral** — don't persist it; use TTLs.
- **Validate early** (notebook 2's pydantic) — untrusted clients will send garbage.

### Further reading
- Figma — [How Figma's multiplayer technology works](https://www.figma.com/blog/how-figmas-multiplayer-technology-works/)
- Martin Kleppmann — [CRDTs: Making ∞ data types eventual-consistency-safe](https://martin.kleppmann.com/papers/crdt-hotos21.pdf)
- Yjs docs — [docs.yjs.dev](https://docs.yjs.dev/)
- Automerge — [automerge.org](https://automerge.org/)
- Liveblocks engineering blog — presence and CRDT patterns
